# Voice and Picture Generation (Open-Source)
Examples for generating images and audio using open-source LLMs and tools.

This demonstrates multimodal output generation using local/open-source alternatives: Stable Diffusion for images, edge-tts for voice.

Note: currently requires Python 3.11 due to compatibility issues with the TTS used.

## Setup and Imports

In [ ]:
# if you haven’t already installed these libraries in the environment:
%pip install transformers torch diffusers accelerate edge-tts nest_asyncio

import torch
from transformers import pipeline
from diffusers import StableDiffusionPipeline
import asyncio
import edge_tts
import nest_asyncio
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

# Apply nest_asyncio to allow asyncio.run() in Jupyter
nest_asyncio.apply()

# Setup output directory (relative to the notebook's location)
output_dir = Path(".") / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load local LLM for text generation (small model for prompts)
text_generator = pipeline("text-generation", model="distilgpt2", device=0 if device == "cuda" else -1)

# Load Stable Diffusion for images (optimized for low VRAM)
image_pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
image_pipe = image_pipe.to(device)
# Enable memory optimizations
image_pipe.enable_attention_slicing()  # Reduces VRAM usage
if device == "cuda":
    image_pipe.enable_xformers_memory_efficient_attention()  # Further optimization (requires xformers)


## LLM-Generated Image Creation (Text-to-Image)

In [ ]:
def generate_image_prompt(theme):
    """Use local LLM to generate a detailed image prompt."""
    prompt_text = f"Create a detailed prompt for an image about {theme}. Describe style, colors, and mood:"
    generated = text_generator(prompt_text, max_length=100, num_return_sequences=1, temperature=0.7)
    return generated[0]['generated_text'].replace(prompt_text, "").strip()


def create_image_from_prompt(prompt, filename="image.png"):
    """Generate image using Stable Diffusion and save to output folder."""
    image = image_pipe(prompt, num_inference_steps=10, guidance_scale=7.5).images[0]
    output_path = output_dir / filename
    image.save(output_path)
    return output_path

# Example usage
theme = "a futuristic city at sunset"
prompt = generate_image_prompt(theme)
print(f"Generated Prompt: {prompt}")

image_path = create_image_from_prompt(prompt)
print(f"Saved image to {image_path}")


## LLM-Generated Voice Creation (Text-to-Speech)

In [ ]:
def generate_voice_script(topic):
    """Use local LLM to generate a short script for TTS."""
    prompt_text = f"Write a 50-100 word script about {topic} for voice narration:"
    generated = text_generator(prompt_text, max_length=150, num_return_sequences=1, temperature=0.7)
    return generated[0]['generated_text'].replace(prompt_text, "").strip()


def create_voice_from_script(script, filename="voice.mp3"):
    """Generate audio using edge-tts and save to output folder."""
    output_path = output_dir / filename
    async def generate():
        voice = edge_tts.Communicate(script, voice="en-US-AriaNeural")
        await voice.save(str(output_path))
    asyncio.run(generate())
    return output_path

# Example usage
topic = "the benefits of renewable energy"
script = generate_voice_script(topic)
print(f"Generated Script: {script}\n")

audio_file = create_voice_from_script(script)
print(f"Saved audio to {audio_file}")
# Play with: import playsound; playsound.playsound(audio_file)


## Combined Example: Story with Image and Voice

In [ ]:
def create_multimodal_story(story_theme):
    """Generate a complete story with image and voice using open-source tools."""
    prompt_text = f"Write a short story about {story_theme}:"
    story_gen = text_generator(prompt_text, max_length=200, num_return_sequences=1, temperature=0.8)
    story = story_gen[0]['generated_text'].replace(prompt_text, "").strip()
    image_prompt = generate_image_prompt(f"Illustrate {story[:50]}...")
    image_path = create_image_from_prompt(image_prompt)
    audio_path = create_voice_from_script(story)
    return story, image_path, audio_path

# Example
theme = "a magical forest adventure"
story, img_path, audio_path = create_multimodal_story(theme)
print(story)
print(img_path, audio_path)
